# Лабораторная работа 13. Основы Torch

PyTorch — один из самых известных фреймворков для машинного обучения и разработки нейросетей на языке Python. Его создала команда Meta AI (Facebook).

Почему его любят: Он позволяет строить нейросети «на лету» (динамические графы), что делает отладку и эксперименты гораздо проще, чем в других инструментах.

Где применяется: В компьютерном зрении, распознавании речи, машинном переводе и генерации текстов (например, на нем работают алгоритмы Instagram и Tesla).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Создание тензоров
# Из списка
t1 = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
print(t1)

tensor([[1., 2.],
        [3., 4.]])


In [3]:
# Случайные числа (нормальное распределение)
t2 = torch.randn(2, 3)
print(t2)

tensor([[1.4321, 0.9723, 1.8819],
        [0.9944, 1.8352, 1.5721]])


In [9]:
# Тензор из нулей
zeros = torch.zeros(2, 2)
# Тензор из единиц
ones = torch.ones(3, 1)
# Единичная матрица
identity = torch.eye(3)
print(identity)

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])


In [13]:
t1 = t1.to(device='cuda')
print(f"Тензор t1:\n{t1}\nТип данных: {t1.dtype}, Устройство: {t1.device}")

Тензор t1:
tensor([[1., 2.],
        [3., 4.]], device='cuda:0')
Тип данных: torch.float32, Устройство: cuda:0


In [14]:
print(f"Тензор t1:\n{t1}\nТип данных: {t1.dtype}, Устройство: {t1.device}")

Тензор t1:
tensor([[1., 2.],
        [3., 4.]], device='cuda:0')
Тип данных: torch.float32, Устройство: cuda:0


In [13]:
# Изменение формы (Reshape & View)
x = torch.arange(12) # [0, 1, 2 ... 11]

print(x)

# view() и reshape() делают похожее,
# но view работает только с непрерывными данными в памяти
x_3x4 = x.view(3, 4)
x_2x2x3 = x.reshape(2, 2, 3)

print(x_2x2x3)

# Использование -1: Torch сам вычислит размерность
x_auto = x.view(2, -1) # Станет (2, 6)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
tensor([[[ 0,  1,  2],
         [ 3,  4,  5]],

        [[ 6,  7,  8],
         [ 9, 10, 11]]])


In [16]:
# Squeeze/Unsqueeze (добавление/удаление единичных размерностей)
y = torch.randn(3, 3)
print(y)
y_unsqueezed = y.unsqueeze(0) # Станет (1, 3, 3) - часто нужно для batch_size
print(y_unsqueezed)
y_final = y_unsqueezed.squeeze() # Уберет все размерности равные 1

print(f"Было: {x.shape}, Стало (view 3x4): {x_3x4.shape}")

tensor([[-0.7749,  1.1753,  0.4373],
        [-0.2235, -0.7339, -1.0586],
        [ 2.8881,  1.0839, -1.1534]])
tensor([[[-0.7749,  1.1753,  0.4373],
         [-0.2235, -0.7339, -1.0586],
         [ 2.8881,  1.0839, -1.1534]]])
Было: torch.Size([12]), Стало (view 3x4): torch.Size([3, 4])


In [17]:
# Математические операции
a = torch.tensor([10, 20, 30])
b = torch.tensor([1, 2, 3])

# Поэлементные операции
print(f"Сложение: {a + b}")
print(f"Умножение на скаляр: {a * 2}")

Сложение: tensor([11, 22, 33])
Умножение на скаляр: tensor([20, 40, 60])


In [18]:
print(a)

tensor([10, 20, 30])


In [16]:
# Матричное умножение (самое важное в Deep Learning)
mat1 = torch.randn(2, 3)
mat2 = torch.randn(3, 4)

# Три способа сделать одно и то же:
res1 = torch.mm(mat1, mat2)
res2 = mat1 @ mat2
res3 = torch.matmul(mat1, mat2) # matmul поддерживает broadcasting (разные размерности)
print(res1)
print(res2)
print(res3)

tensor([[ 1.7763, -1.0586,  0.7602,  2.1329],
        [ 3.0893,  1.2203, -2.1118,  0.7786]])
tensor([[ 1.7763, -1.0586,  0.7602,  2.1329],
        [ 3.0893,  1.2203, -2.1118,  0.7786]])
tensor([[ 1.7763, -1.0586,  0.7602,  2.1329],
        [ 3.0893,  1.2203, -2.1118,  0.7786]])


In [17]:
# Агрегация и оси
m = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

print(f"Общая сумма: {m.sum()}")
print(f"Сумма по столбцам (dim=0): {m.sum(dim=0)}")
print(f"Среднее по строкам (dim=1): {m.mean(dim=1)}")
print(f"Индекс максимального элемента: {m.argmax()}")

Общая сумма: 21.0
Сумма по столбцам (dim=0): tensor([5., 7., 9.])
Среднее по строкам (dim=1): tensor([2., 5.])
Индекс максимального элемента: 5


In [18]:
# Слайсинг и индексация
z = torch.randn(4, 4)
print(f"Первая строка: {z[0, :]}")
print(f"Второй столбец: {z[:, 1]}")
print(f"Подматрица 2x2 в центре:\n{z[1:3, 1:3]}")

Первая строка: tensor([-0.2903, -0.5893, -1.2136, -0.3792])
Второй столбец: tensor([-0.5893, -0.1327,  0.5967,  0.6388])
Подматрица 2x2 в центре:
tensor([[-0.1327,  0.3127],
        [ 0.5967,  0.1054]])


In [ ]:
# Работа с GPU (CUDA)
if torch.cuda.is_available():
    device = torch.device("cuda")
    z = z.to(device) # Перенос на видеокарту
    print("Тензор перенесен на GPU")
else:
    print("GPU недоступен, работаем на CPU")

In [19]:
# Соединение тензоров
t_a = torch.ones(2, 2)
t_b = torch.zeros(2, 2)

# Конкатенация (склейка)
cat_v = torch.cat([t_a, t_b], dim=0) # По вертикали (4, 2)
cat_h = torch.cat([t_a, t_b], dim=1) # По горизонтали (2, 4)

# Стек (создание новой размерности)
stacked = torch.stack([t_a, t_b]) # Станет (2, 2, 2)

print(f"Shape после stack: {stacked.shape}")

Shape после stack: torch.Size([2, 2, 2])


Перцептрон

In [20]:
# Подготовка данных
iris = load_iris()
X, y = iris.data, iris.target

# Берем только 2 класса для классического перцептрона (бинарная классификация)
X = X[y != 2]
y = y[y != 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
# Масштабирование — критически важно для нейросетей
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Конвертация в тензоры PyTorch
X_train_t = torch.FloatTensor(X_train)
print(X_train.shape)
y_train_t = torch.FloatTensor(y_train).reshape(-1, 1)
print(X_train_t.shape)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).reshape(-1, 1)

(80, 4)
torch.Size([80, 4])


In [22]:
# Реализация Перцептрона
class Perceptron(nn.Module):
    def __init__(self, input_dim):
        super(Perceptron, self).__init__()
        self.fc = nn.Linear(input_dim, 1) # Один слой
        self.sigmoid = nn.Sigmoid()       # Функция активации для вероятности

    def forward(self, x):
        return self.sigmoid(self.fc(x))

In [24]:
model = Perceptron(input_dim=4)
criterion = nn.BCELoss() # Binary Cross Entropy для бинарной задачи
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [25]:
# Цикл обучения
for epoch in range(100):
    optimizer.zero_grad()           # Обнуляем градиенты
    outputs = model(X_train_t)      # Forward pass
    loss = criterion(outputs, y_train_t)
    loss.backward()                 # Backward pass (расчет градиентов)
    optimizer.step()                # Обновление весов

    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')

Epoch [20/100], Loss: 0.2200
Epoch [40/100], Loss: 0.1145
Epoch [60/100], Loss: 0.0780
Epoch [80/100], Loss: 0.0595
Epoch [100/100], Loss: 0.0483


In [26]:
# Проверка
with torch.no_grad():
    predictions = model(X_test_t).round()
    accuracy = (predictions == y_test_t).float().mean()
    print(f'Accuracy: {accuracy:.2f}')

Accuracy: 1.00


Multi-Output сеть с разными активациями

In [27]:
class MultiOutputNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(MultiOutputNet, self).__init__()

        # Общий скрытый слой
        self.hidden = nn.Linear(input_size, hidden_size)

        # Выход 1: Регрессия (например, цена объекта)
        self.reg_layer = nn.Linear(hidden_size, 1)

        # Выход 2: Классификация (например, тип объекта)
        self.clf_layer = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Применяем ReLU в скрытом слое (отсекаем отрицательные значения)
        x = F.relu(self.hidden(x))

        # Выход регрессии. Часто без активации или Tanh/Sigmoid, если диапазон ограничен.
        # Оставим линейным для произвольных чисел.
        reg_output = self.reg_layer(x)

        # Выход классификации. Используем Softmax для распределения вероятностей.
        clf_output = F.softmax(self.clf_layer(x), dim=1)

        return reg_output, clf_output

In [28]:
net = MultiOutputNet(input_size=10, hidden_size=20, num_classes=3)

In [29]:
# Создаем фейковые входные данные (batch_size=5, features=10)
dummy_input = torch.randn(5, 10)

# Получаем результаты
regression_res, classification_res = net(dummy_input)

print("Результат регрессии (числа):\n", regression_res.detach().numpy())
print("\nРезультат классификации (вероятности):\n", classification_res.detach().numpy())

Результат регрессии (числа):
 [[0.06601365]
 [0.01313621]
 [0.239047  ]
 [0.4642796 ]
 [0.04988956]]

Результат классификации (вероятности):
 [[0.43302882 0.25759912 0.30937213]
 [0.37224916 0.2936766  0.33407414]
 [0.3365393  0.33709615 0.32636452]
 [0.25912282 0.29023355 0.45064366]
 [0.36193147 0.34936738 0.28870115]]


# Лабораторная работа
Легенда:
Вернёмся к датасету California House Pricing. Вам нужно создать нейросеть, которая по этим признакам одновременно:
Предсказывает рыночную стоимость (задача регрессии).
Определяет категорию жилья (эконом, стандарт, люкс — задача классификации, для этого необходимо создать вторую целевую переменную, определив пороги для разбиения на классы жилья и преобразовав данные).

Техническое задание:
Входной слой: 10 нейронов.
Скрытый слой: 64 нейрона. Используйте функцию активации ReLU.
Выход 1 (Regression Head): 1 нейрон. Используйте Identity (без активации) или ReLU, если цена не может быть отрицательной.
Выход 2 (Classification Head): 3 нейрона. Используйте Softmax для получения распределения вероятностей.
